In [39]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder
from sklearn.model_selection import train_test_split
import pickle 



In [40]:
data=pd.read_csv("Churn_Modelling.csv")

In [41]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [42]:
data=data.drop(columns=["RowNumber","CustomerId","Surname"],axis=1,inplace=False)


In [43]:
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [44]:
## Encode categorical variables 

label_encoder=LabelEncoder()
data['Gender']=label_encoder.fit_transform(data['Gender'])


In [45]:
## One-hot encode 'Geography' column

one_hot_encoded_geo=OneHotEncoder(handle_unknown='ignore')

geo_encoded=one_hot_encoded_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=one_hot_encoded_geo.get_feature_names_out(['Geography']))

In [46]:
geo_encoded

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [47]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [48]:
## split the data into featuers and target 
X=data.drop('EstimatedSalary',axis=1)
y=data['EstimatedSalary']

In [49]:
## split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


## scale the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [50]:
## Save the encoders and scaler for future use
with open('reg_label_encoder.pkl','wb') as f:
    pickle.dump(label_encoder,f)

with open('reg_one_hot_encoder.pkl','wb') as f:
    pickle.dump(one_hot_encoded_geo,f)

with open('reg_scaler.pkl','wb') as f: 
    pickle.dump(scaler,f)
    

## ANN Regression Problem Statement

In this project, we are building an Artificial Neural Network (ANN) to predict salary based on various features such as age, experience, education level, and other relevant attributes. The dataset contains numerical and categorical features that need to be preprocessed before feeding into the neural network. The goal is to train a regression model that can accurately predict continuous salary values from these input features.

In [51]:
import tensorflow as tf
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense 



In [52]:
## Build the model 

model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1)
])


## compile the model 
model.compile(optimizer='adam',loss='mean_squared_error',metrics=['mae'])

In [53]:
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_12 (Dense)            (None, 64)                832       
                                                                 
 dense_13 (Dense)            (None, 32)                2080      
                                                                 
 dense_14 (Dense)            (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [54]:
from tensorflow.keras.callbacks import EarlyStopping ,TensorBoard

import datetime

## setup tensorboard 
log_dir="regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callbackk=TensorBoard(log_dir=log_dir,histogram_freq=1)



In [56]:
## Setup Early stopping 

early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,
restore_best_weights=True)


In [57]:
history=model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,callbacks=[early_stopping_callback,tensorboard_callbackk])

Epoch 1/100
250/250 [==============================] - 0s 564us/step - loss: 13384301568.0000 - mae: 100381.0938 - val_loss: 13010304000.0000 - val_mae: 98534.0703
Epoch 2/100
250/250 [==============================] - 0s 582us/step - loss: 13244028928.0000 - mae: 99681.3359 - val_loss: 12727604224.0000 - val_mae: 97101.3359
Epoch 3/100
250/250 [==============================] - 0s 409us/step - loss: 12730014720.0000 - mae: 97142.3672 - val_loss: 11983501312.0000 - val_mae: 93328.0234
Epoch 4/100
250/250 [==============================] - 0s 413us/step - loss: 11700215808.0000 - mae: 92036.5625 - val_loss: 10722291712.0000 - val_mae: 86955.7266
Epoch 5/100
250/250 [==============================] - 0s 419us/step - loss: 10184923136.0000 - mae: 84517.4141 - val_loss: 9071962112.0000 - val_mae: 78609.3047
Epoch 6/100
250/250 [==============================] - 0s 434us/step - loss: 8391267328.0000 - mae: 75525.5938 - val_loss: 7299918848.0000 - val_mae: 69727.8750
Epoch 7/100
250/250 [===

In [32]:
%load_ext tensorboard

In [35]:
%tensorboard --logdir regressionlogs/fit 

Reusing TensorBoard on port 6007 (pid 5614), started 0:03:50 ago. (Use '!kill 5614' to kill it.)

In [37]:
test_loss,test_mae=model.evaluate(X_test,y_test)
print(f"Test Loss: {test_loss}, Test MAE: {test_mae}")

63/63 [==============================] - 0s 235us/step - loss: 3354891776.0000 - mae: 50094.4805
Test Loss: 3354891776.0, Test MAE: 50094.48046875


In [38]:
model.save('regression_model.h5')

/Users/anupdangi/Desktop/AnupAI/Research/DS_ML_DL_NLP_BOOTCAMP/venv/lib/python3.9/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1
